# Lab: Experiment Tracking & Model Registry with MLflow**Week 4 | MLOps Learning Course**---## Operator ContextYou know how to:- Push images to a container registry with tags- Promote deployments from staging → production- Roll back a bad deploy with `kubectl rollout undo`This lab teaches you the **exact same workflow, but for ML models.**MLflow is to models what ECR/Harbor + your CI pipeline is to services.By the end of this lab you will:1. Train multiple models with different hyperparameters2. Log every training run (params, metrics, artifacts)3. Compare runs side-by-side (like comparing canary deploys)4. Register the best model in the registry5. Promote it through lifecycle stages6. Roll back to a previous version

---## Step 1: Setup & Imports> **Ops parallel:** This is your `FROM python:3.11` — setting up the base environment.

In [ ]:
import mlflowimport mlflow.sklearnfrom mlflow.tracking import MlflowClientimport pandas as pdimport numpy as npfrom sklearn.datasets import load_winefrom sklearn.model_selection import train_test_splitfrom sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifierfrom sklearn.linear_model import LogisticRegressionfrom sklearn.metrics import accuracy_score, f1_score, classification_reportimport matplotlib.pyplot as pltimport warningswarnings.filterwarnings('ignore')print(f"MLflow version: {mlflow.__version__}")print("All imports successful — ready to track experiments.")

---## Step 2: Configure MLflow Tracking> **Ops parallel:** This is like setting your `DOCKER_REGISTRY` env var — telling the system where to store artifacts.>> MLflow stores everything locally by default in an `mlruns/` directory. In production, you'd point this at a remote server (like pointing Docker at ECR instead of local daemon).

In [ ]:
# Set up MLflow to track locally (file-based store)# In production, you'd use: mlflow.set_tracking_uri("http://mlflow-server:5000")mlflow.set_tracking_uri("file:./mlruns")# Create an experiment — this is like creating a new CI pipelineEXPERIMENT_NAME = "wine-quality-classification"mlflow.set_experiment(EXPERIMENT_NAME)print(f"Tracking URI: {mlflow.get_tracking_uri()}")print(f"Experiment: {EXPERIMENT_NAME}")print()print("Think of this experiment as a GitHub Actions workflow.")print("Every time we train a model, it's a 'run' — like a workflow execution.")

---## Step 3: Prepare the Data> **Ops parallel:** This is your source code checkout step. The data is your raw material, just like source code is the raw material for a Docker build.

In [ ]:
# Load the Wine dataset (3-class classification problem)wine = load_wine()X = pd.DataFrame(wine.data, columns=wine.feature_names)y = wine.target# Split into train/test — same split every time for reproducibilityX_train, X_test, y_train, y_test = train_test_split(    X, y, test_size=0.2, random_state=42)print(f"Dataset: Wine Quality Classification")print(f"Features: {X.shape[1]}")print(f"Classes: {len(set(y))} ({wine.target_names})")print(f"Training samples: {X_train.shape[0]}")print(f"Test samples: {X_test.shape[0]}")print()print("Note: We fix random_state=42 for reproducibility.")print("In real MLOps, you'd also version the dataset itself (DVC, Delta Lake, etc).")

---## Step 4: Train Models & Log Experiments> **Ops parallel:** Each training run below is like a CI build. We log everything — parameters (build args), metrics (health checks), and artifacts (the built image).>> After this step, you'll have multiple runs to compare — like comparing builds from different branches or configs.### 4a: Define the training function with MLflow logging

In [ ]:
def train_and_log(model, model_name, params, X_train, X_test, y_train, y_test):    """    Train a model and log everything to MLflow.        This is the equivalent of a CI pipeline that:    1. Takes build args (params)    2. Builds the artifact (trains the model)    3. Runs health checks (evaluates metrics)    4. Pushes to registry (logs to MLflow)    """    with mlflow.start_run(run_name=model_name):        # Log parameters — these are your build args        for key, value in params.items():            mlflow.log_param(key, value)        mlflow.log_param("model_type", model_name)                # Train the model — this is the 'docker build' step        model.fit(X_train, y_train)                # Evaluate — these are your post-deploy health checks / SLIs        y_pred = model.predict(X_test)        accuracy = accuracy_score(y_test, y_pred)        f1 = f1_score(y_test, y_pred, average='weighted')                # Log metrics — like reporting SLIs after a canary deploy        mlflow.log_metric("accuracy", accuracy)        mlflow.log_metric("f1_score", f1)        mlflow.log_metric("training_samples", len(X_train))                # Log the model artifact — like pushing the Docker image        mlflow.sklearn.log_model(model, "model")                # Log a classification report as a text artifact        report = classification_report(y_test, y_pred, target_names=wine.target_names)        with open("classification_report.txt", "w") as f:            f.write(report)        mlflow.log_artifact("classification_report.txt")                run_id = mlflow.active_run().info.run_id        print(f"✅ Run logged: {model_name}")        print(f"   Run ID: {run_id[:8]}...")        print(f"   Accuracy: {accuracy:.4f}")        print(f"   F1 Score: {f1:.4f}")        print(f"   (Params, metrics, and model artifact all stored)")        print()                return run_id, accuracy, f1print("Training function defined. Each call = one pipeline run.")print("We log: params (build args), metrics (SLIs), artifacts (the model).")

### 4b: Run Multiple Experiments> **Ops parallel:** This is like running your pipeline with different Dockerfile configurations to see which produces the best image (smallest, fastest, most secure).

In [ ]:
# Experiment 1: Logistic Regression (simple baseline)run1_id, run1_acc, run1_f1 = train_and_log(    model=LogisticRegression(max_iter=1000, C=1.0, random_state=42),    model_name="logistic-regression-baseline",    params={"max_iter": 1000, "C": 1.0, "solver": "lbfgs"},    X_train=X_train, X_test=X_test,    y_train=y_train, y_test=y_test)# Experiment 2: Random Forest (moderate complexity)run2_id, run2_acc, run2_f1 = train_and_log(    model=RandomForestClassifier(n_estimators=100, max_depth=10, random_state=42),    model_name="random-forest-v1",    params={"n_estimators": 100, "max_depth": 10, "min_samples_split": 2},    X_train=X_train, X_test=X_test,    y_train=y_train, y_test=y_test)# Experiment 3: Random Forest (more trees, deeper)run3_id, run3_acc, run3_f1 = train_and_log(    model=RandomForestClassifier(n_estimators=200, max_depth=20, random_state=42),    model_name="random-forest-v2-deeper",    params={"n_estimators": 200, "max_depth": 20, "min_samples_split": 2},    X_train=X_train, X_test=X_test,    y_train=y_train, y_test=y_test)# Experiment 4: Gradient Boosting (high complexity)run4_id, run4_acc, run4_f1 = train_and_log(    model=GradientBoostingClassifier(n_estimators=150, learning_rate=0.1, max_depth=5, random_state=42),    model_name="gradient-boosting-v1",    params={"n_estimators": 150, "learning_rate": 0.1, "max_depth": 5},    X_train=X_train, X_test=X_test,    y_train=y_train, y_test=y_test)# Experiment 5: Gradient Boosting (tuned)run5_id, run5_acc, run5_f1 = train_and_log(    model=GradientBoostingClassifier(n_estimators=200, learning_rate=0.05, max_depth=4, random_state=42),    model_name="gradient-boosting-v2-tuned",    params={"n_estimators": 200, "learning_rate": 0.05, "max_depth": 4},    X_train=X_train, X_test=X_test,    y_train=y_train, y_test=y_test)print("=" * 50)print("All 5 experiments logged. Each one is a versioned,")print("reproducible record — like 5 tagged Docker images")print("with full build provenance.")

---## Step 5: Compare Experiments> **Ops parallel:** This is like looking at your CI dashboard to compare build times, test pass rates, and image sizes across runs. You're picking the best candidate for production.

In [ ]:
# Query all runs from our experiment — like listing all builds in a pipelineclient = MlflowClient()experiment = client.get_experiment_by_name(EXPERIMENT_NAME)runs = client.search_runs(    experiment_ids=[experiment.experiment_id],    order_by=["metrics.f1_score DESC"])# Display comparison tableprint("=" * 70)print(f"{'Run Name':<30} {'Accuracy':<12} {'F1 Score':<12} {'Model Type':<20}")print("=" * 70)for run in runs:    name = run.info.run_name    acc = run.data.metrics.get("accuracy", 0)    f1 = run.data.metrics.get("f1_score", 0)    model_type = run.data.params.get("model_type", "unknown")    print(f"{name:<30} {acc:<12.4f} {f1:<12.4f} {model_type:<20}")print("=" * 70)print()best_run = runs[0]print(f"🏆 Best model: {best_run.info.run_name}")print(f"   F1 Score: {best_run.data.metrics['f1_score']:.4f}")print(f"   Run ID: {best_run.info.run_id}")print()print("This is your 'canary winner' — the build you'd promote to production.")

### 5b: Visualize the Comparison> Like a Grafana dashboard comparing canary metrics across deploys.

In [ ]:
# Plot metrics comparisonrun_names = [run.info.run_name for run in runs]accuracies = [run.data.metrics["accuracy"] for run in runs]f1_scores = [run.data.metrics["f1_score"] for run in runs]fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))# Accuracy comparisonbars1 = ax1.barh(run_names, accuracies, color='steelblue')ax1.set_xlabel('Accuracy')ax1.set_title('Model Accuracy Comparison')ax1.set_xlim(0.8, 1.0)for bar, val in zip(bars1, accuracies):    ax1.text(val + 0.002, bar.get_y() + bar.get_height()/2, f'{val:.4f}', va='center')# F1 Score comparisonbars2 = ax2.barh(run_names, f1_scores, color='darkorange')ax2.set_xlabel('F1 Score (weighted)')ax2.set_title('Model F1 Score Comparison')ax2.set_xlim(0.8, 1.0)for bar, val in zip(bars2, f1_scores):    ax2.text(val + 0.002, bar.get_y() + bar.get_height()/2, f'{val:.4f}', va='center')plt.tight_layout()plt.savefig("experiment_comparison.png", dpi=100, bbox_inches='tight')plt.show()print("Chart saved as experiment_comparison.png")print("In the MLflow UI, you get this comparison for free — plus interactive filtering.")

---## Step 6: Register the Best Model> **Ops parallel:** This is the moment you push your image to the registry and tag it. You're saying "this artifact is a release candidate.">> Before this step, the model was just a run artifact. After registration, it has a name, a version number, and a lifecycle stage — just like `myapp:v2.1.0` in your container registry.

In [ ]:
# Register the best model in the Model RegistryMODEL_NAME = "wine-quality-classifier"best_run_id = best_run.info.run_idmodel_uri = f"runs:/{best_run_id}/model"# Register the model — like `docker push myregistry/myapp:v1`result = mlflow.register_model(model_uri, MODEL_NAME)print(f"✅ Model registered!")print(f"   Name: {result.name}")print(f"   Version: {result.version}")print(f"   Source Run: {best_run_id[:8]}...")print(f"   Stage: {result.current_stage}")print()print("The model now has a version number and lifecycle stage.")print("It's like pushing to ECR — the artifact is now referenceable by name + version.")

### Register a second model version (for rollback demo later)> We register two versions so we can demonstrate rollback — like having v1 and v2 in your registry.

In [ ]:
# Register the second-best model as another versionsecond_best_run = runs[1]second_model_uri = f"runs:/{second_best_run.info.run_id}/model"result2 = mlflow.register_model(second_model_uri, MODEL_NAME)print(f"✅ Second model version registered!")print(f"   Name: {result2.name}")print(f"   Version: {result2.version}")print(f"   Source Run: {second_best_run.info.run_name}")print(f"   F1 Score: {second_best_run.data.metrics['f1_score']:.4f}")print()print("Now we have two versions in the registry — like having :v1 and :v2 tags.")print("Let's promote them through stages.")

---## Step 7: Promote Through Lifecycle Stages> **Ops parallel:** This is your deployment pipeline: staging → production.>> ```> None → Staging → Production → Archived> ```>> Just like you wouldn't deploy straight to prod without testing in staging, models go through the same gate process.

In [ ]:
import timeclient = MlflowClient()# Promote Version 1 to Staging first# This is like deploying to your staging environmentclient.transition_model_version_stage(    name=MODEL_NAME,    version="1",    stage="Staging")print("📋 Version 1 → Staging")print("   (Like deploying to staging for integration tests)")print()time.sleep(1)# After 'testing passes', promote to Production# This is like your staging → prod promotion after QA sign-offclient.transition_model_version_stage(    name=MODEL_NAME,    version="1",    stage="Production")print("🚀 Version 1 → Production")print("   (Like promoting from staging to prod after validation)")print()# Now promote Version 2 to Stagingclient.transition_model_version_stage(    name=MODEL_NAME,    version="2",    stage="Staging")print("📋 Version 2 → Staging")print("   (New version in staging while v1 serves production)")print()# Show current stateprint("=" * 50)print("Current Model Registry State:")print("=" * 50)for mv in client.search_model_versions(f"name='{MODEL_NAME}'"):    print(f"  Version {mv.version}: Stage={mv.current_stage}, Run={mv.run_id[:8]}...")print()print("This is your 'kubectl get deployments' equivalent for models.")print("You can see what's in each environment at a glance.")

---## Step 8: Load a Production Model (Serving)> **Ops parallel:** This is like `docker pull myregistry/myapp:production` — loading the artifact that's currently tagged for production use.

In [ ]:
# Load the model that's currently in Production# This is how a serving system would pull the modelprod_model = mlflow.pyfunc.load_model(f"models:/{MODEL_NAME}/Production")# Make predictions with the production modelsample_data = X_test[:5]predictions = prod_model.predict(sample_data)print("🏭 Loaded Production model and made predictions:")print()for i, (pred, actual) in enumerate(zip(predictions, y_test[:5])):    status = "✅" if pred == actual else "❌"    print(f"   Sample {i+1}: Predicted={wine.target_names[pred]}, "          f"Actual={wine.target_names[actual]} {status}")print()print("In production, your serving layer (FastAPI, Seldon, KServe) would")print("do exactly this: load_model('models:/model-name/Production').")print("When you promote a new version, the serving layer picks it up.")

---## Step 9: Rollback — Load a Previous Version> **Ops parallel:** This is `kubectl rollout undo deployment/myapp` — reverting to the previous good version.>> Scenario: Version 2 was promoted to production but is causing issues. We need to roll back to Version 1.### Why this matters in banking:> A fraud model that's too aggressive blocks legitimate transactions. A model that's too lenient lets fraud through. Either way, you need to roll back in minutes, not hours.

In [ ]:
# SCENARIO: Version 2 was promoted to Production but it's causing issues.# We need to roll back to Version 1.print("⚠️  INCIDENT: Model v2 in production is causing elevated false positives")print("   Action: Rolling back to v1")print()# Demote v2 back to Stagingclient.transition_model_version_stage(    name=MODEL_NAME,    version="2",    stage="Staging",    archive_existing_versions=False)# Ensure v1 is in Productionclient.transition_model_version_stage(    name=MODEL_NAME,    version="1",    stage="Production",    archive_existing_versions=False)print("🔄 Rollback complete:")print("   Version 1: Production (restored)")print("   Version 2: Staging (demoted for investigation)")print()# Verify by loading the production model againrolled_back_model = mlflow.pyfunc.load_model(f"models:/{MODEL_NAME}/Production")predictions_after = rolled_back_model.predict(X_test[:5])print("✅ Production model is now serving Version 1 again.")print("   Predictions after rollback match the previous v1 outputs.")print()print("=" * 50)print("KEY TAKEAWAY:")print("=" * 50)print("Rollback was instant because we had:")print("  ✓ Both versions stored in the registry")print("  ✓ Lifecycle stages tracked")print("  ✓ Full provenance (which run, which params, which metrics)")print()print("Without a model registry, rollback means:")print("  ✗ 'Who has the old model file?'")print("  ✗ 'What hyperparameters did we use?'")print("  ✗ 'Can someone retrain it? What data did we use?'")print("  ✗ Meanwhile, the bad model is still serving traffic.")

---## Step 10: View Full Experiment Lineage> **Ops parallel:** This is like running `docker inspect` on an image — seeing the full provenance of an artifact.

In [ ]:
# Get full details of the production modelprint("=" * 60)print("PRODUCTION MODEL — Full Lineage Report")print("=" * 60)print()# Get the production version detailsprod_versions = client.get_latest_versions(MODEL_NAME, stages=["Production"])prod_version = prod_versions[0]# Get the source runsource_run = client.get_run(prod_version.run_id)print(f"Model Name:      {prod_version.name}")print(f"Version:         {prod_version.version}")print(f"Stage:           {prod_version.current_stage}")print(f"Source Run ID:   {prod_version.run_id[:16]}...")print(f"Run Name:        {source_run.info.run_name}")print(f"Creation Time:   {pd.Timestamp(prod_version.creation_timestamp, unit='ms')}")print()print("Parameters (Build Args):")for key, value in source_run.data.params.items():    print(f"  {key}: {value}")print()print("Metrics (SLIs):")for key, value in source_run.data.metrics.items():    print(f"  {key}: {value}")print()print("This is your full audit trail. In a regulated environment (banking!),")print("this answers: 'What model is in production, and how was it built?'")print("Same question as 'What image is deployed, and what's in it?'")

---## Step 11: Cleanup & MLflow UI> **Ops parallel:** Like checking your Kubernetes dashboard after a deploy to see the full picture.

In [ ]:
# Clean up the temporary fileimport osif os.path.exists("classification_report.txt"):    os.remove("classification_report.txt")print("=" * 60)print("LAB COMPLETE")print("=" * 60)print()print("To explore your experiments in the MLflow UI:")print()print("  1. Open a terminal in this directory")print("  2. Run: mlflow ui --port 5000")print("  3. Open: http://localhost:5000")print()print("In the UI you can:")print("  - Compare runs side-by-side (like Grafana dashboards)")print("  - View artifacts for each run")print("  - See the model registry with stages")print("  - View parameter and metric history")print()print("=" * 60)print("SUMMARY: What You Just Did (in Ops Terms)")print("=" * 60)print()print("  ML Action                    Ops Equivalent")print("  ─────────────────────────    ─────────────────────────────────")print("  Set experiment name          Create CI pipeline")print("  Train with mlflow.start_run  Execute pipeline run")print("  Log params                   Store build args / env vars")print("  Log metrics                  Report SLIs / health checks")print("  Log model artifact           Push image to registry")print("  Compare experiments          Compare canary deploys")print("  Register model               Tag image as release candidate")print("  Promote to Production        Deploy to prod (with approval)")print("  Rollback version             kubectl rollout undo")print()print("Next week: Model Serving & Deployment Patterns")